# MFC decomposition data

Compute and export the full ENSO-related lower-tropospheric moisture-flux-convergence decomposition: thermodynamic (TH), moisture-convergence dynamic (MCD), transient eddy (TE), and the surface-pressure term (S). Plotting is intentionally kept out of this notebook.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

OUTPUT_DIR = Path('/Users/rizzie/Work/PaperENSO/Scripts_v2/Fig3').resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

UVQ_PATH = Path('/Users/rizzie/ClimateData/era5-monthly/uvq_1980-2025_1000-1hpa.nc')
SP_PATH = Path('/Users/rizzie/ClimateData/era5-monthly/surface_presure_DJF_1980-2026.nc')
NINO34_PATH = Path('/Users/rizzie/ClimateData/climate_index/ENSO/nino34.anom.csv')
OUT_CSV = OUTPUT_DIR / 'dekomposisi_mfc_data.csv'

# Differentiate on this buffer, then area-average only over TARGET_BOX.
BUFFER_BOX = {'lon_min': 90.0, 'lon_max': 130.0, 'lat_min': -10.0, 'lat_max': 6.0}
TARGET_BOX = {'lon_min': 95.0, 'lon_max': 125.0, 'lat_min': -6.0, 'lat_max': 2.0}
START = '1980-12-01'
END = '2025-02-28'
FULL_YEARS = np.arange(1981, 2026)
P1_YEARS = np.arange(1981, 2007)
P2_YEARS = np.arange(2007, 2026)
DJF_MONTHS = (12, 1, 2)

G = 9.80665
R_EARTH = 6_371_000.0


In [2]:
def djf_seasonal_mean(da):
    """Compute DJF means, assigning December to the following year."""
    da = da.sel(time=slice(START, END))
    month_mask = da.time.dt.month.isin(DJF_MONTHS)
    djf_year = xr.where(da.time.dt.month == 12, da.time.dt.year + 1, da.time.dt.year)
    da = da.sel(time=month_mask).assign_coords(
        djf_year=('time', djf_year.sel(time=month_mask).data)
    )
    return da.sel(time=da.djf_year.isin(FULL_YEARS)).groupby('djf_year').mean('time')


def regress_1d(y, x, dim='djf_year'):
    """Ordinary-least-squares slope of y onto x along dim."""
    y, x = xr.align(y, x, join='inner')
    x_anom = x - x.mean(dim)
    y_anom = y - y.mean(dim)
    return (y_anom * x_anom).mean(dim) / x_anom.var(dim)


def vertical_integrate_700_to_surface(term, sp):
    """Trapezoidally integrate from 700 hPa to local surface pressure.

    Each pressure interval uses its actual thickness.  In the lowest valid interval,
    the flux is linearly interpolated to local ps, so no below-ground portion is used.
    """
    if 700.0 not in term.level.values:
        raise ValueError('The 700 hPa level is required for this integration.')
    # The source pressure coordinate is descending (1000 to 1 hPa), so use a
    # coordinate-independent mask to retain only the 700 hPa–surface column.
    term = term.where((term.level >= 700.0) & (term.level <= 1000.0), drop=True)
    term = term.sortby('level').chunk({'level': -1})
    pressure_pa = term.level.values.astype(float) * 100.0
    layer = np.arange(pressure_pa.size - 1)

    lower = term.isel(level=slice(None, -1)).rename({'level': 'layer'}).assign_coords(layer=layer)
    upper = term.isel(level=slice(1, None)).rename({'level': 'layer'}).assign_coords(layer=layer)
    p_lower = xr.DataArray(pressure_pa[:-1], coords={'layer': layer}, dims='layer')
    p_upper = xr.DataArray(pressure_pa[1:], coords={'layer': layer}, dims='layer')

    # dp is zero below ground and is truncated to ps in the surface-crossing layer.
    dp = (sp - p_lower).clip(min=0.0, max=p_upper - p_lower)
    fraction = dp / (p_upper - p_lower)
    flux_at_top = lower + fraction * (upper - lower)
    contribution = 0.5 * (lower + flux_at_top) * dp
    # Discard only intervals with zero local thickness, not entire pressure levels.
    contribution = contribution.where(dp > 0.0, 0.0).where(np.isfinite(sp))
    return contribution.sum('layer', skipna=False) / G


def mfc_per_level(q_u, q_v):
    """Return pressure-level MFC before vertical integration."""
    q_u, q_v = xr.align(q_u, q_v, join='inner')
    q_u, q_v = q_u.sortby('lat'), q_v.sortby('lat')
    lat_rad = np.deg2rad(q_u.lat)
    cos_lat = np.cos(lat_rad)
    dqu_dlon = q_u.differentiate('lon') * (180.0 / np.pi)
    dqvcos_dlat = (q_v * cos_lat).differentiate('lat') * (180.0 / np.pi)
    return (-(dqu_dlon + dqvcos_dlat) / (R_EARTH * cos_lat)).rename('mfc_per_level')


def value_at_surface(term, sp):
    """Interpolate a lower-tropospheric field to local ps using every 700–1000 hPa level."""
    lower_troposphere = term.where(
        (term.level >= 700.0) & (term.level <= 1000.0), drop=True
    ).sortby('level')
    return lower_troposphere.interp(level=sp).fillna(term.sel(level=1000.0))


def surface_pressure_term(q, u, v, sp, nino):
    """Return δS = δ(qs Vs · ∇ps) / g, evaluated at the local surface."""
    q_s = value_at_surface(q, sp)
    u_s = value_at_surface(u, sp)
    v_s = value_at_surface(v, sp)
    sp = sp.sortby('lat')
    lat_rad = np.deg2rad(sp.lat)
    dps_dlon = sp.differentiate('lon') * (180.0 / np.pi) / (R_EARTH * np.cos(lat_rad))
    dps_dlat = sp.differentiate('lat') * (180.0 / np.pi) / R_EARTH
    surface_flux = q_s * (u_s * dps_dlon + v_s * dps_dlat)
    return (regress_1d(surface_flux, nino) / G).rename('surface_pressure')


In [3]:
# Load the combined ERA5 file and normalize its coordinate names.
uvq_ds = xr.open_dataset(UVQ_PATH, chunks={'valid_time': 3})[['q', 'u', 'v']]
uvq_ds = uvq_ds.rename({'valid_time': 'time', 'pressure_level': 'level',
                        'latitude': 'lat', 'longitude': 'lon'})
uvq_ds = uvq_ds.assign_coords(lon=uvq_ds.lon % 360).sortby('lon')

lat_slice = (slice(BUFFER_BOX['lat_min'], BUFFER_BOX['lat_max'])
             if uvq_ds.lat.values[0] < uvq_ds.lat.values[-1]
             else slice(BUFFER_BOX['lat_max'], BUFFER_BOX['lat_min']))
uvq_ds = uvq_ds.sel(
    time=slice(START, END),
    lat=lat_slice,
    lon=slice(BUFFER_BOX['lon_min'], BUFFER_BOX['lon_max']),
).sortby('lat')

q, u, v = uvq_ds.q, uvq_ds.u, uvq_ds.v

sp_ds = xr.open_dataset(SP_PATH, chunks={'valid_time': 3})[['sp']]
sp_ds = sp_ds.rename({'valid_time': 'time', 'latitude': 'lat', 'longitude': 'lon'})
sp_ds = sp_ds.assign_coords(lon=sp_ds.lon % 360).sortby('lon').sel(
    time=slice(START, END), lat=lat_slice,
    lon=slice(BUFFER_BOX['lon_min'], BUFFER_BOX['lon_max']),
).sortby('lat')
sp = sp_ds.sp
print('All available pressure levels (hPa):', q.level.values.tolist())
print('Buffer q shape:', q.shape)
print('Buffer sp shape:', sp.shape, '| units:', sp.attrs.get('units', 'unknown'))


/tmp/ipykernel_33345/1397021359.py:2: UserWarning: The specified chunks separate the stored chunks along dimension "valid_time" starting at index 3. This could degrade performance. Instead, consider rechunking after loading.
  uvq_ds = xr.open_dataset(UVQ_PATH, chunks={'valid_time': 3})[['q', 'u', 'v']]


All available pressure levels (hPa): [1000.0, 975.0, 950.0, 925.0, 900.0, 875.0, 850.0, 825.0, 800.0, 775.0, 750.0, 700.0, 650.0, 600.0, 550.0, 500.0, 450.0, 400.0, 350.0, 300.0, 250.0, 225.0, 200.0, 175.0, 150.0, 125.0, 100.0, 70.0, 50.0, 30.0, 20.0, 10.0, 7.0, 5.0, 3.0, 2.0, 1.0]
Buffer q shape: (135, 37, 65, 161)
Buffer sp shape: (135, 65, 161) | units: Pa


/tmp/ipykernel_33345/1397021359.py:18: UserWarning: The specified chunks separate the stored chunks along dimension "valid_time" starting at index 3. This could degrade performance. Instead, consider rechunking after loading.
  sp_ds = xr.open_dataset(SP_PATH, chunks={'valid_time': 3})[['sp']]


In [4]:
q_djf, u_djf, v_djf, sp_djf = (djf_seasonal_mean(da) for da in (q, u, v, sp))
q_djf, u_djf, v_djf, sp_djf = xr.align(q_djf, u_djf, v_djf, sp_djf, join='exact')

nino_df = pd.read_csv(NINO34_PATH, parse_dates=['Date'])
nino_columns = [column for column in nino_df.columns if column != 'Date']
if len(nino_columns) != 1:
    raise ValueError(f'Expected one Niño3.4 value column, found: {nino_columns}')
nino_column = nino_columns[0]
nino_df = nino_df.set_index('Date').loc[START:END].copy()
nino_df[nino_column] = pd.to_numeric(nino_df[nino_column], errors='coerce').replace(-99.99, np.nan)
nino_df = nino_df[nino_df.index.month.isin(DJF_MONTHS)]
nino_df['djf_year'] = nino_df.index.year + (nino_df.index.month == 12).astype('int8')

nino_djf = nino_df.groupby('djf_year')[nino_column].mean().reindex(FULL_YEARS)
if nino_djf.isna().any():
    raise ValueError(f'Missing DJF Niño3.4 values for: {nino_djf.index[nino_djf.isna()].tolist()}')
nino_djf = xr.DataArray(nino_djf.to_numpy(), coords={'djf_year': nino_djf.index.to_numpy()},
                         dims='djf_year', name='nino34')
nino_djf_std = (nino_djf - nino_djf.mean('djf_year')) / nino_djf.std('djf_year', ddof=0)

print('DJF years in ERA5:', q_djf.sizes['djf_year'])
print('DJF years in surface pressure:', sp_djf.sizes['djf_year'])
print('DJF years in Niño3.4:', nino_djf_std.sizes['djf_year'])


DJF years in ERA5: 45
DJF years in surface pressure: 45
DJF years in Niño3.4: 45


In [5]:
period_results = {}
for period_name, years in {'P1': P1_YEARS, 'P2': P2_YEARS}.items():
    q_p, u_p, v_p, sp_p, n_p = xr.align(
        q_djf.sel(djf_year=years), u_djf.sel(djf_year=years),
        v_djf.sel(djf_year=years), sp_djf.sel(djf_year=years),
        nino_djf_std.sel(djf_year=years), join='inner'
    )
    qbar, ubar, vbar = (field.mean('djf_year') for field in (q_p, u_p, v_p))
    spbar = sp_p.mean('djf_year')  # Pa; period-mean surface for the local column geometry.
    q_anom, u_anom, v_anom = q_p - qbar, u_p - ubar, v_p - vbar

    flux_components = {
        'total': (regress_1d(q_p * u_p, n_p), regress_1d(q_p * v_p, n_p)),
        'dynamic': (qbar * regress_1d(u_anom, n_p), qbar * regress_1d(v_anom, n_p)),
        'thermodynamic': (ubar * regress_1d(q_anom, n_p), vbar * regress_1d(q_anom, n_p)),
        # δTE: transient-eddy/nonlinear moisture-flux response.
        'nonlinear': (regress_1d(q_anom * u_anom, n_p), regress_1d(q_anom * v_anom, n_p)),
    }

    # Integrate convergence itself so the variable lower boundary remains explicit.
    volume_mfc = {name: vertical_integrate_700_to_surface(
        mfc_per_level(fu, fv), spbar
    ) for name, (fu, fv) in flux_components.items()}
    surface_mfc = surface_pressure_term(q_p, u_p, v_p, sp_p, n_p)
    # Full decomposition: total = TH + MCD + TE − S.
    component_mfc = {
        'total': volume_mfc['total'] - surface_mfc,
        'dynamic': volume_mfc['dynamic'],
        'thermodynamic': volume_mfc['thermodynamic'],
        'nonlinear': volume_mfc['nonlinear'],
        'surface_pressure': -surface_mfc,  # plotted contribution is −δS
    }
    # Residual after TH and MCD: δTE − δS in the full decomposition.
    residual_mfc = (component_mfc['total'] - component_mfc['dynamic']
                    - component_mfc['thermodynamic'])
    # Derivatives were computed on BUFFER_BOX; restrict and compute every term together.
    all_mfc = {**component_mfc, 'residual': residual_mfc}
    target_mfc = xr.Dataset({
        name: mfc.sel(
            lat=slice(TARGET_BOX['lat_min'], TARGET_BOX['lat_max']),
            lon=slice(TARGET_BOX['lon_min'], TARGET_BOX['lon_max']),
        ) for name, mfc in all_mfc.items()
    }).compute()
    weights = np.cos(np.deg2rad(target_mfc.lat))
    component_area_mean = {
        name: float(mfc.weighted(weights).mean(('lat', 'lon')) )
        for name, mfc in target_mfc.data_vars.items()
    }
    period_results[period_name] = component_area_mean
    print(f"{period_name} residual (δTE − δS): {component_area_mean['residual']:+.4e} kg m^-2 s^-1")


P1 residual (δTE − δS): +5.8583e-07 kg m^-2 s^-1


P2 residual (δTE − δS): +2.0960e-07 kg m^-2 s^-1


In [6]:
term_order = ['total', 'dynamic', 'thermodynamic', 'nonlinear', 'residual', 'surface_pressure']
df_out = pd.DataFrame({
    'P1_kg_m2_s': [period_results['P1'][term] for term in term_order],
    'P2_kg_m2_s': [period_results['P2'][term] for term in term_order],
}, index=term_order)
df_out['diff_P2_minus_P1_kg_m2_s'] = df_out['P2_kg_m2_s'] - df_out['P1_kg_m2_s']
df_out.to_csv(OUT_CSV, index_label='term')
print(f'Saved computation output -> {OUT_CSV}')
df_out


Saved computation output -> /Users/rizzie/Work/PaperENSO/Scripts_v2/Fig3/dekomposisi_mfc_data.csv


,P1_kg_m2_s,P2_kg_m2_s,diff_P2_minus_P1_kg_m2_s
total,7.243035e-07,7.977121e-06,7.252818e-06
dynamic,-5.029041e-07,6.082427e-06,6.585331e-06
thermodynamic,6.413774e-07,1.685096e-06,1.043718e-06
nonlinear,-2.954929e-07,-4.765501e-07,-1.810571e-07
residual,5.858302e-07,2.095989e-07,-3.762313e-07
surface_pressure,8.813231e-07,6.861490e-07,-1.951741e-07
